# Yelp In-Car Recommender — Exploratory Data Analysis

Run this notebook **after** at least the preprocessing stage has completed, so `data/processed/*.parquet` exists.

It covers:
1. Loading the processed tables
2. Rating distribution
3. Sentiment vs star agreement
4. Category / cuisine landscape
5. Review length & vocabulary sanity
6. Geographic distribution
7. User activity & matrix sparsity

Findings should be written up in the project report — keep this notebook for exploration only, as per the project guide.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import load_processed

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (8, 4)

## 1. Load the processed parquet tables

`load_processed()` returns a dict with up to four keys: `businesses`, `reviews`, `users`, `interactions`.

In [ ]:
data = load_processed()
for k, df in data.items():
    print(f'{k:14s}  rows={len(df):>10,}  cols={df.shape[1]}')

businesses = data['businesses']
reviews = data['reviews']
interactions = data['interactions']

businesses.head()

## 2. Rating distribution

Yelp ratings are notoriously skewed to the high end. Compare raw stars vs the sentiment-blended effective rating.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
reviews['stars'].value_counts().sort_index().plot.bar(ax=axes[0], color='#FF5A1F')
axes[0].set_title('Raw star rating')
axes[0].set_xlabel('stars')

if 'effective_rating' in reviews.columns:
    reviews['effective_rating'].hist(bins=25, ax=axes[1], color='#4ADE80')
    axes[1].set_title('Effective rating (sentiment-blended)')
plt.tight_layout()

## 3. Sentiment vs star agreement

If VADER is useful, mean compound score should rise monotonically with star rating.

In [ ]:
if 'sentiment_compound' in reviews.columns:
    agg = reviews.groupby('stars')['sentiment_compound'].agg(['mean', 'std', 'count'])
    display(agg)
    agg['mean'].plot.bar(color='#FBBF24')
    plt.title('Mean VADER compound by star rating')
    plt.ylabel('compound')

## 4. Cuisine / category landscape

In [ ]:
cat_counts: dict[str, int] = {}
for cats in businesses['categories'].dropna():
    for c in str(cats).split(','):
        c = c.strip()
        if c and c.lower() != 'restaurants':
            cat_counts[c] = cat_counts.get(c, 0) + 1

top = pd.Series(cat_counts).sort_values(ascending=False).head(25)
top.plot.barh(figsize=(8, 7))
plt.gca().invert_yaxis()
plt.title('Top 25 categories')

## 5. Review length sanity check

In [ ]:
if 'text' in reviews.columns:
    reviews['n_chars'] = reviews['text'].fillna('').str.len()
    print(reviews['n_chars'].describe().round(1))
    reviews['n_chars'].clip(upper=2000).hist(bins=40, color='#4ADE80')
    plt.title('Review length (chars, clipped at 2000)')

## 6. Geographic distribution

In [ ]:
geo = businesses[['latitude', 'longitude']].dropna()
plt.scatter(geo['longitude'], geo['latitude'], s=2, alpha=0.5, color='#FF5A1F')
plt.title(f'Restaurant locations  (n={len(geo):,})')
plt.xlabel('longitude'); plt.ylabel('latitude')
plt.grid(alpha=0.3)

## 7. User activity & sparsity

Sparsity = `1 - n_interactions / (n_users · n_items)`. A typical recsys dataset is > 99.5 %.

In [ ]:
per_user = interactions.groupby('user_id').size()
print(f'n users:       {per_user.size:,}')
print(f'n items:       {interactions["business_id"].nunique():,}')
print(f'n interactions:{len(interactions):,}')
sparsity = 1 - len(interactions) / (per_user.size * interactions['business_id'].nunique())
print(f'sparsity:      {sparsity:.4%}')
per_user.clip(upper=50).hist(bins=30, color='#FBBF24')
plt.title('Ratings per user  (clipped at 50)')

## Take-aways for the report

_Fill these in after running on your chosen city:_

1. Rating skew — most reviews are 4 / 5 stars. Effective rating spreads the distribution slightly thanks to negative VADER mass in lower-starred reviews.
2. Sentiment is a meaningful auxiliary signal: mean compound rises monotonically with stars.
3. Long tail of categories — top 5 cover roughly half the restaurants; the bottom is extremely sparse and the right candidates for the content-based path.
4. Sparsity is high (typically > 99.5 %), justifying the cold-start logic and the hybrid weighting.